**⚠️ Before we start:** run `update` in the terminal, then execute these two commands to move the models we will use to the current directory:
```
cp /home/max/cosmo_models/pet-mad-s-v1.5.0.pt /home/max/materials/day-09-uncertainty-flashmd/nc-and-flashmd/
cp /home/max/cosmo_models/flashmd-16.pt /home/max/materials/day-09-uncertainty-flashmd/nc-and-flashmd/
```

## Machine learning for molecular dynamics beyond potentials

Welcome! In this tutorial, we will explore different simulation strategies for molecular dynamics (MD), focusing on their performance and stability. We will compare standard **conservative** forces, **non-conservative** forces, **Multiple-Time-Step (MTS)** integration combining the two, and the **FlashMD** method.

Let's start by importing some necessary libraries.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ase.io
import chemiscope

In [ ]:
# (also, this is how you would download the two models we will use)
# (not necessary now because we already have them on the virtual machine)

# import upet
# upet.save_upet(model="pet-mad", size="s", version="1.5.0", output="pet-mad-s-v1.5.0.pt")

# from flashmd import get_pretrained
# _, flashmd_model = get_pretrained("pet-omatpes-v2", 16)
# flashmd_model.save(f"flashmd-{16}.pt")

### Defining Simulation Steps

We want to simulate a total of 64 fs of liquid water dynamics across all methods. Note the difference in required integration step sizes:
- **Standard MD** requires small 0.5 fs steps (128 steps total) for stability.
- **MTS (Multiple Time Step)** uses an outer step of 4 fs (16 steps).
- **FlashMD** allows for 16 fs time steps (only 4 steps needed!).

In [ ]:
# total of 64 fs for all simulations:

NORMAL_STEPS = 128  # for simulations with 0.5 fs steps
MTS_STEPS = 16      # the MTS simulation has an outer step of 4 fs
FLASHMD_STEPS = 4   # FlashMD does steps of 16 fs

### Initial Configuration

Before running LAMMPS, we need a starting structure. We'll load a liquid water configuration, wrap the atoms into our periodic simulation box, and export it as a LAMMPS data file.

In [ ]:
water = ase.io.read("water.xyz")
water.wrap()
ase.io.write(
    "water.data",
    water,
    format="lammps-data",
    atom_style="atomic",
    units="metal",
    specorder=["O", "H"],
    masses=True,
)

### Performance Metric

To effectively compare the computational cost of these different approaches, we'll define a helper function to calculate the simulation throughput in nanoseconds per day (ns/day). Higher is better!

In [ ]:
def ns_per_day(simulated_time_fs: float, elapsed_seconds: float) -> float:
    return simulated_time_fs * 1e-6 * 24 * 60 * 60 / elapsed_seconds


def read_lammps(simulation_type, num_steps) -> float:
    # run_command(f"lmp -in {input_file}")
    with open(f"lammps-{simulation_type}.log") as f:
        header_index = None
        for i, line in enumerate(f):
            if line.strip().startswith("Time"):
                header_index = i
                break
    if header_index is None:
        raise ValueError("Could not find header line in log.lammps")

    data = np.loadtxt(f"lammps-{simulation_type}.log", skiprows=header_index + 1, max_rows=num_steps + 1)
    time_fs = data[:, 0]
    potential_energy = data[:, 2]  # total energy column
    kinetic_energy = data[:, 3]  # total energy column
    total_energy = data[:, 4]  # total energy column
    data = data[3:]  # timings (skip first 3 rows due to warmup)
    elapsed = data[-1, -1] - data[0, -1]
    simulated_time_fs = data[-1, 0] - data[0, 0]
    throughput = ns_per_day(simulated_time_fs, elapsed)

    return {
        "throughput": throughput,
        "time": time_fs,
        "potential_energy": potential_energy,
        "kinetic_energy": kinetic_energy,
        "total_energy": total_energy,
    }

Let's initialize a dictionary to store the parsed results (like energy, time, and throughput) from each of our simulations.

In [ ]:
results = {}

### 1. Conservative Force Simulation

Standard machine-learning interatomic potentials are **conservative**: the forces are derived as the exact negative gradient of a potential energy surface (PES). This guarantees energy conservation but it comes at a computational cost.

**⚠️ Terminal Execution Required:**
LAMMPS must be run from your terminal to generate the data for this cell. Open your terminal in this directory and execute:
```bash
lmp -in lammps-c.in -log lammps-c.log
```
This runs a conservative simulation. Once the simulation is complete, run the cell below to load the results.

In [ ]:
# after running the conservative simulation:
results_c = read_lammps("c", NORMAL_STEPS)
print("Conservative throughput:", results_c["throughput"], "ns/day")
results["conservative"] = results_c

Let's visualize the energy conservation of our standard conservative model. We plot the fluctuations in potential, kinetic, and total energy. You should observe that the total energy is relatively flat and well-conserved.

In [ ]:
plt.plot(results_c["time"], results_c["potential_energy"] - results_c["potential_energy"].mean(), label="Potential energy")
plt.plot(results_c["time"], results_c["kinetic_energy"] - results_c["kinetic_energy"].mean(), label="Kinetic energy")
plt.plot(results_c["time"], results_c["total_energy"] - results_c["total_energy"].mean(), label="Total energy")
plt.legend()
plt.xlabel("Time (fs)")
plt.ylabel("Energy (eV)")

We will define a utility function to read the trajectory files (`.lammpstrj`) generated by LAMMPS so we can analyze and visualize the atomic motions.

In [ ]:
def read_lammps_trajectory(filename: str) -> list:
    frames = ase.io.read(filename, ":")
    symbols = water.get_chemical_symbols()
    for atoms in frames:
        atoms.set_chemical_symbols(symbols)
        atoms.pbc = True
    return frames

structures_c = read_lammps_trajectory("conservative.lammpstrj")

def chemiscope_show(structures, results):
    return chemiscope.show(
        structures,
        properties={
            "time": results["time"],
            "potential energy": results["potential_energy"],
        },
        settings={"structure": [{"unitCell": True}]},
    )
        
chemiscope_show(structures_c, results["conservative"])

### 2. Non-Conservative Force Simulation

Recent studies have explored **non-conservative** machine learning potentials. Instead of deriving forces from an energy surface, these models predict atomic forces directly. This can improve computational efficiency and local force accuracy, but the resulting forces are not conservative and the system may therefore exhibit energy drift over time.

**⚠️ Terminal Execution Required:**
Run the following command in your terminal:
```bash
lmp -in lammps-nc.in -log lammps-nc.log
```
After this non-conservative simulation finishes, run the cell below.

In [ ]:
# after running the non-conservative simulation:
results_nc = read_lammps("nc", NORMAL_STEPS)
print("Non-conservative throughput:", results_nc["throughput"], "ns/day")
results["non-conservative"] = results_nc

Let's compare the total energy of our conservative and non-conservative runs. The non-conservative model may show a noticeable drift in total energy since it doesn't strictly adhere to energy conservation laws.

In [ ]:
plt.plot(results["conservative"]["time"], results["conservative"]["total_energy"], label="conservative")
plt.plot(results["non-conservative"]["time"], results["non-conservative"]["total_energy"], label="non-conservative")
plt.xlabel("Time (fs)")
plt.ylabel("Total Energy (eV/atom)")

### 3. Multiple Time Step (MTS) Simulation

To speed up simulations without sacrificing stability, we can employ a **Multiple Time Step (MTS)** approach. MTS evaluates rapidly changing short-range forces frequently, but updates slowly changing slow forces less often, allowing for a larger effective outer time step. In this case, the rapidly changing force is taken to be the non-conservative force, while the slow-changing forces are a correction term given by the difference between the conservative and non-conservative forces.

**⚠️ Terminal Execution Required:**
Run the following command in your terminal:
```bash
lmp -in lammps-mts.in -log lammps-mts.log
```

In [ ]:
# after running MTS:
results_mts = read_lammps("mts", MTS_STEPS)
print("MTS throughput:", results_mts["throughput"], "ns/day")
results["mts"] = results_mts

Let's compare the energy conservation of the MTS simulation against the previous conservative and non-conservative runs. The MTS algorithm conserves the energy while running significantly faster than a conservative simulation.

In [ ]:
plt.plot(results["conservative"]["time"], results["conservative"]["total_energy"]-results["conservative"]["total_energy"][0], label="Conservative")
plt.plot(results["non-conservative"]["time"], results["non-conservative"]["total_energy"]-results["non-conservative"]["total_energy"][0], label="Non-conservative")
plt.plot(results["mts"]["time"], results["mts"]["total_energy"]-results["mts"]["total_energy"][0], label="MTS")
plt.legend()
plt.xlabel("Time (fs)")
plt.ylabel("Total Energy (eV)")

### 4. FlashMD Simulation

**FlashMD** is an approach designed to push the boundaries of simulation speed. By learning to predict the state of the system multiple steps into the future, it allows for very large integration time steps (e.g., 16 fs for water) while remaining stable. This provides a large boost to simulation throughput.

**⚠️ Terminal Execution Required:**
Run the following command in your terminal:
```bash
lmp -in lammps-flashmd.in -log lammps-flashmd.log
```

In [ ]:
# after running FlashMD:
results_flashmd = read_lammps("flashmd", FLASHMD_STEPS)
print("FlashMD throughput:", results_flashmd["throughput"], "ns/day")
results["flashmd"] = results_flashmd

Now let's use **Chemiscope** to interactively visualize the trajectory produced by FlashMD. You can rotate the 3D structure and observe the water dynamics in action.

In [ ]:
structures_flashmd = read_lammps_trajectory("flashmd.lammpstrj")
chemiscope_show(structures_flashmd, results["flashmd"])

### 5. Experiment: Do Something Crazy!

In [ ]:
from metatomic.torch import load_atomistic_model
flashmd_model = load_atomistic_model("flashmd-16.pt")

Now it's your turn to play around! Since the FlashMD model is universal, you can simulate something crazy. Here are various systems you can add to the water simulation: you just need to uncomment and run. You can also try further changes, like setting up the water system at an extreme temperature or run for a longer time.

Modify and repeat the following two cells to try different things.

In [ ]:
import ase.build
import ase.units
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
from flashmd.ase.velocity_verlet import VelocityVerlet

# ----- MOLECULES ------ #
atoms = ase.build.molecule("NH3")  # chill
# atoms = ase.build.molecule("bicyclobutane")  # funny
# atoms = ase.build.molecule("biphenyl")  # still stable

# ----- CRAZY ------ #
# atoms = ase.build.molecule("C60")  # kaboom
# atoms = ase.build.nanotube(3, 3) * (1, 1, 6)  # YOLO

# ----- RANDOM ELEMENTS ------ #
# atoms = ase.Atoms(numbers=np.random.randint(1, 81, (5,)), positions=np.random.randn(5, 3))  # random

atoms.positions += [5.1, 6.6, 8.1]
crazy_water = water + atoms
crazy_water.wrap()

# visualize the prepared structure
chemiscope.show(crazy_water, mode="structure", settings={"structure": [{"unitCell": True}]})

In [ ]:
# The structure is now ready. Run FlashMD:

MaxwellBoltzmannDistribution(crazy_water, temperature_K=300)  # you can change the T!

trajectory = []
integrator = VelocityVerlet(
    atoms=crazy_water,
    timestep=16*ase.units.fs,
    model=flashmd_model,
    rescale_energy=False,
)
integrator.attach(lambda: trajectory.append(crazy_water.copy()))
integrator.attach(lambda: print("step", integrator.nsteps))
integrator.run(20)  # run for 20 steps, you can change this!

chemiscope.show(trajectory, mode="structure", settings={"structure": [{"unitCell": True}]})

Now re-run the last two cells again with a different system!

### Explore the Web Viewers

**⚠️ Copy links into a browser outside the VM for optimal performance**

Before you go, make sure to check out these three online viewers. They are great tools for exploring machine learning potentials and visualizing molecular dynamics trajectories in the browser.

1. Lightweight (runs **PET-MAD-XS** in your browser):
https://peterspackman.github.io/mlip.cpp/

2. Heavyweight (runs **UMA** on remote GPUs):
https://aidemos.atmeta.com/uma

3. Explore many different MLIPs! (including **PET-MAD with uncertainty quantification**):
https://huggingface.co/spaces/ManasSharma07/mlip-playground